# Clase Práctica de Ansible en Google Colab
Este cuaderno contiene una guía completa paso a paso para aprender a configurar Ansible,
comprender el concepto de idempotencia y desplegar un servidor web Nginx localmente.

Desarrollado para ser convertido a formato Jupyter (.ipynb) usando **Jupytext**.

### Paso 1: Instalación y Configuración del Entorno
Esta celda prepara la máquina virtual de Colab instalando Ansible, el servidor OpenSSH 
y configurando los accesos por clave para que Ansible pueda gestionarse a sí mismo (`localhost`).

In [ ]:
# 1. Actualizar el sistema e instalar OpenSSH
!apt-get update -y
!apt-get install -y openssh-server
# Instalar Ansible vía pip para evitar incompatibilidades de versión con los paquetes del sistema
!pip install ansible

# 2. Generar claves SSH y autorizarlas localmente si no existen (evita prompts interactivos)
![ -f /root/.ssh/id_rsa ] || ssh-keygen -q -t rsa -N '' -f /root/.ssh/id_rsa
!grep -qF "$(cat /root/.ssh/id_rsa.pub 2>/dev/null)" /root/.ssh/authorized_keys 2>/dev/null || cat /root/.ssh/id_rsa.pub >> /root/.ssh/authorized_keys
!grep -q "localhost" /root/.ssh/known_hosts 2>/dev/null || ssh-keyscan -H localhost >> /root/.ssh/known_hosts

# 3. Iniciar el servicio SSH obligatorio
!service ssh start

# 4. Verificar que Ansible se comunica correctamente (Debe responder con "pong")
!ansible localhost -m ping

### Paso 2: Crear el Archivo de Inventario
Ansible necesita un inventario para saber a qué servidores apuntar. 
Usamos `%%writefile` para crear un archivo llamado `hosts` que define el grupo `[webservers]`.

In [ ]:
%%writefile hosts
[webservers]
localhost ansible_connection=local

### Paso 3: Crear el Archivo HTML Personalizado
Crearemos la estructura de la página web que nuestro servidor va a servir. 
Este archivo será copiado al directorio del servidor web mediante Ansible.

In [ ]:
%%writefile index.html
<!DOCTYPE html>
<html>
<head>
    <title>Clase de Ansible en Colab</title>
    <style>
        body { font-family: Arial, sans-serif; text-align: center; margin-top: 50px; background-color: #f4f4f9; }
        h1 { color: #2c3e50; }
        p { color: #7f8c8d; font-size: 18px; }
    </style>
</head>
<body>
    <h1>¡Hola Mundo desde Ansible!</h1>
    <p>Esta página fue desplegada de forma 100% automatizada usando un Playbook en Google Colab.</p>
</body>
</html>

### Paso 4: Definir el Playbook de Ansible
El Playbook describe el estado deseado de nuestra infraestructura. 
Contiene 3 tareas secuenciales e independientes.

In [ ]:
%%writefile playbook_nginx.yml
---
- name: Configurar y Desplegar Servidor Web Nginx
  hosts: webservers
  tasks:

    - name: 1. Asegurar que Nginx esté instalado
      apt:
        name: nginx
        state: present
        update_cache: yes

    - name: 2. Copiar el archivo index.html personalizado al servidor
      copy:
        src: index.html
        dest: /var/www/html/index.html
        owner: www-data
        group: www-data
        mode: '0644'

    - name: 3. Asegurar que Nginx esté iniciado y habilitado
      service:
        name: nginx
        state: started
        enabled: yes

### Paso 5: Ejecutar el Playbook de Ansible
Corremos el playbook especificando nuestro inventario con el parámetro `-i`.

**Tip pedagógico:** Ejecuta esta celda una vez y observa las tareas marcadas en amarillo (`changed`). 
Ejecútala una segunda vez de inmediato: verás todo en verde (`ok`), demostrando la **idempotencia**.

In [ ]:
!ansible-playbook -i hosts playbook_nginx.yml

### Paso 6: Validar el Despliegue Local
Como estamos dentro del entorno controlado de Google Colab sin IP pública directa, 
validamos el éxito del despliegue haciendo una petición HTTP interna con `curl`.

In [ ]:
!curl http://localhost